# Create LaTeX Results Tables

Generate booktabs-style LaTeX tables of the test set results for every task (columns) and
every model (rows).

Four tables are produced: classification and segmentation results for both full fine-tuning
and head-only training. The generated tables require `\usepackage{booktabs}`,
`\usepackage{siunitx}` and `\usepackage{graphicx}` in the document preamble.

In [ ]:
from pathlib import Path

import pandas as pd

MODEL_DISPLAY_NAMES = {
    "deblurmim_convmae": "Deblurring ConvMAE",
    "deblurmim_mae": "Deblurring MAE",
    "dinov3_convnext_base": "DINOv3 ConvNeXt-B",
    "dinov3_convnext_large": "DINOv3 ConvNeXt-L",
    "dinov3_vitb16": "DINOv3 ViT-B",
    "dinov3_vitl16": "DINOv3 ViT-L",
    "echocare": "EchoCare",
    "openus": "OpenUS",
    "samus": "SAMUS",
    "tinyusfm": "TinyUSFM",
    "ultrafedfm": "UltraFedFM",
    "ultrasam": "UltraSAM",
    "urfm_base": "URFM-B",
    "urfm_large": "URFM-L",
    "usfm": "USFM",
    "usfmae": "USF-MAE",
}

# Short task labels, used as column headers so that the tables fit the page.
TASK_DISPLAY = {
    # Classification
    "aul_mass_classification": "AUL Mass",
    "busbra_histology_classification": "BUS-BRA Hist.",
    "busbra_pathology_classification": "BUS-BRA Path.",
    "camus_classification": "CAMUS",
    "covid_blues_classification": "COVID-BLUES",
    "fetal_planes_brain_plane_classification": "Fetal Brain",
    "fetal_planes_plane_classification": "Fetal Plane",
    "gbcu_classification": "GBCU",
    "masld_classification": "MASLD",
    "mmotu_classification": "MMOTU",
    "tn3k_classification": "TN3K",
    # Segmentation
    "aul_liver_segmentation": "AUL Liver",
    "aul_mass_segmentation": "AUL Mass",
    "busbra_segmentation": "BUS-BRA",
    "camus_segmentation": "CAMUS",
    "mmotu_segmentation": "MMOTU",
    "open_kidney_capsule_segmentation": "OK Capsule",
    "open_kidney_regions_segmentation": "OK Regions",
    "psfhs_segmentation": "PSFHS",
    "stmus_segmentation": "STMUS",
    "tn3k_segmentation": "TN3K",
}

PATHO_CLS_TASKS = [
    "aul_mass_classification",
    "busbra_histology_classification",
    "busbra_pathology_classification",
    "covid_blues_classification",
    "gbcu_classification",
    "masld_classification",
    "mmotu_classification",
    "tn3k_classification",
]
NONPATHO_CLS_TASKS = [
    "camus_classification",
    "fetal_planes_brain_plane_classification",
    "fetal_planes_plane_classification",
]
PATHO_SEG_TASKS = [
    "aul_mass_segmentation",
    "busbra_segmentation",
    "mmotu_segmentation",
    "tn3k_segmentation",
]
ANATOMICAL_SEG_TASKS = [
    "aul_liver_segmentation",
    "camus_segmentation",
    "open_kidney_capsule_segmentation",
    "open_kidney_regions_segmentation",
    "psfhs_segmentation",
    "stmus_segmentation",
]

# Only full shot results are tabulated.
SHOT_PCT = 1.0

# The siunitx setup for the metric columns. `mode = text` and
# `reset-text-series = false` are what let the font change on the best value in
# a column reach the digits; `detect-weight = true` is the siunitx v2 spelling
# of the same thing. `retain-zero-uncertainty` keeps the "+/- 0.000" on the
# cells whose seeds agree exactly, which siunitx otherwise discards.
SISETUP_OPTIONS = [
    "table-format = 1.3(3)",
    "uncertainty-mode = separate",
    "retain-zero-uncertainty = true",
    "mode = text",
    "reset-text-series = false",
]

# `\textbf` selects Computer Modern's bold *extended* face, whose digits are
# 15% wider than the regular face, so an emboldened cell is wider than the rest
# of its column and siunitx cannot align it. Series "b" is bold but not
# extended, and has exactly the regular digit widths, as long as both come from
# the 10pt design size -- hence setting the size explicitly rather than with
# `\footnotesize`. The tables are scaled to the text width afterwards, so this
# is not the size the reader sees.
BOLD_SERIES = r"\fontseries{b}\selectfont"
TABLE_FONT_SIZE = r"\fontsize{10}{12}\selectfont"

## Load Aggregated Test Results

Load the `test_results.csv` file generated by `aggregate_results.ipynb`.

In [ ]:
test_results = pd.read_csv("test_results_20260831.csv")

test_results["accuracy"] = test_results.apply(
    lambda row: (
        row["seq_accuracy"] if not pd.isna(row["seq_accuracy"]) else row["accuracy"]
    ),
    axis=1,
)
test_results["balanced_accuracy"] = test_results.apply(
    lambda row: (
        row["seq_balanced_accuracy"]
        if not pd.isna(row["seq_balanced_accuracy"])
        else row["balanced_accuracy"]
    ),
    axis=1,
)
test_results["f1_macro"] = test_results.apply(
    lambda row: (
        row["seq_f1_macro"] if not pd.isna(row["seq_f1_macro"]) else row["f1_macro"]
    ),
    axis=1,
)
test_results["f1_weighted"] = test_results.apply(
    lambda row: (
        row["seq_f1_weighted"]
        if not pd.isna(row["seq_f1_weighted"])
        else row["f1_weighted"]
    ),
    axis=1,
)
test_results["mcc"] = test_results.apply(
    lambda row: row["seq_mcc"] if not pd.isna(row["seq_mcc"]) else row["mcc"],
    axis=1,
)

test_results.head()

## Summarize the Test Results

Compute the mean and standard deviation of each metric over the random seeds. The decoder
prefix is stripped from `mode` so that the results are grouped by training paradigm alone
(the segmentation decoder depends on the backbone architecture).

In [ ]:
def summarize_results(results: pd.DataFrame, metric: str) -> pd.DataFrame:
    """Summarize a metric over the random seeds of each run configuration.

    Args:
        results: The aggregated test results, with a `paradigm` column.
        metric: The name of the metric column to summarize.

    Returns:
        A DataFrame with one row per task, model and paradigm, and `mean`
        and `std` columns holding the summary statistics over the seeds.
    """
    return (
        results[results["shot_pct"] == SHOT_PCT]
        .groupby(["task", "model", "paradigm"])[metric]
        .agg(["mean", "std"])
        .reset_index()
    )


test_results["paradigm"] = test_results["mode"].str.replace(
    r"^.*?_(full_finetuning|head_only)$", r"\1", regex=True
)

classification_summary = summarize_results(
    test_results[test_results["task"].str.contains("classification")], "f1_macro"
)
segmentation_summary = summarize_results(
    test_results[test_results["task"].str.contains("segmentation")], "positive_dice"
)

classification_summary.head()

## Build the LaTeX Tables

Each table lists the models as rows and the tasks as columns, with the task columns grouped
by category. The best result for each task is shown in bold.

In [ ]:
def format_cell(mean: float, std: float, is_best: bool) -> str:
    """Format a mean and standard deviation as a siunitx table cell.

    Args:
        mean: The mean value of the metric.
        std: The standard deviation of the metric.
        is_best: Whether this is the best value in its column.

    Returns:
        The cell contents in siunitx uncertainty notation, prefixed with the
        bold font series if it is the best value in its column. The prefix is
        deliberately not braced, which would stop siunitx aligning the cell.
    """
    cell = f"{mean:.3f}({std:.3f})"

    return f"{BOLD_SERIES} {cell}" if is_best else cell


def build_table_frame(
    summary: pd.DataFrame, paradigm: str, task_groups: dict[str, list[str]]
) -> pd.DataFrame:
    """Pivot a summary of the results into a table of formatted LaTeX cells.

    Args:
        summary: A summary of the results, as returned by `summarize_results`.
        paradigm: The training paradigm to tabulate, e.g. `"full_finetuning"`.
        task_groups: A mapping of category name to the ordered list of tasks it
            contains. Determines both the column groups and the column order.

    Returns:
        A DataFrame of LaTeX cell strings indexed by model display name, with a
        two-level column index of (category, task display name).

    Raises:
        ValueError: If any requested task has no results for the paradigm.
    """
    tasks = [task for group in task_groups.values() for task in group]
    subset = summary[(summary["paradigm"] == paradigm) & summary["task"].isin(tasks)]

    missing = set(tasks) - set(subset["task"])
    if missing:
        raise ValueError(f"No '{paradigm}' results for tasks: {sorted(missing)}")

    models = [model for model in MODEL_DISPLAY_NAMES if model in set(subset["model"])]
    means = subset.pivot(index="model", columns="task", values="mean").loc[
        models, tasks
    ]
    stds = subset.pivot(index="model", columns="task", values="std").loc[models, tasks]
    best_models = means.idxmax()

    return pd.DataFrame(
        [
            [
                format_cell(
                    means.at[model, task],
                    stds.at[model, task],
                    best_models[task] == model,
                )
                for task in tasks
            ]
            for model in models
        ],
        index=[MODEL_DISPLAY_NAMES[model] for model in models],
        columns=pd.MultiIndex.from_tuples(
            [
                (category, TASK_DISPLAY[task])
                for category, group in task_groups.items()
                for task in group
            ]
        ),
    )


def insert_cmidrules(tabular: str, columns: pd.Index) -> str:
    r"""Insert booktabs rules underneath the column group headers.

    Pandas emits `\multicolumn` headers for grouped columns but no accompanying
    `\cmidrule`s, so they are added here.

    Args:
        tabular: The LaTeX tabular environment produced by `Styler.to_latex`.
        columns: The two-level column index of the tabulated DataFrame.

    Returns:
        The tabular environment with a rule under each column group header.

    Raises:
        ValueError: If the tabular environment has no column group header row.
    """
    categories = columns.get_level_values(0)
    rules = []
    column = 2  # The first column holds the model names.
    for category in dict.fromkeys(categories):
        width = sum(category == other for other in categories)
        rules.append(rf"\cmidrule(lr){{{column}-{column + width - 1}}}")
        column += width

    lines = tabular.splitlines()
    header_lines = [i for i, line in enumerate(lines) if r"\multicolumn" in line]
    if not header_lines:
        raise ValueError("No column group header found in the tabular environment.")
    lines.insert(header_lines[0] + 1, " ".join(rules))

    return "\n".join(lines)


def to_booktabs_latex(frame: pd.DataFrame, caption: str, label: str) -> str:
    """Render a table of LaTeX cells as a booktabs-style table environment.

    Args:
        frame: A DataFrame of cell strings from `build_table_frame`.
        caption: The caption of the table.
        label: The label used to reference the table.

    Returns:
        The complete LaTeX source of the table.
    """
    tabular = frame.style.hide(axis="index", names=True).to_latex(
        hrules=True,
        siunitx=True,
        multicol_align="c",
        column_format="l" + "S" * frame.shape[1],
    )
    sisetup = "\n".join(
        [r"\sisetup{", *(f"  {option}," for option in SISETUP_OPTIONS), "}"]
    )

    return "\n".join(
        [
            r"\begin{table}[t]",
            r"\centering",
            rf"\caption{{{caption}}}",
            rf"\label{{{label}}}",
            TABLE_FONT_SIZE,
            r"\setlength{\tabcolsep}{3pt}",
            sisetup,
            r"\resizebox{\textwidth}{!}{%",
            insert_cmidrules(tabular.strip(), frame.columns),
            r"}",
            r"\end{table}",
            "",
        ]
    )

## Generate the Tables

Write each table to a `.tex` file alongside this notebook and print its source.

In [ ]:
CLS_TASK_GROUPS = {
    "Pathological": PATHO_CLS_TASKS,
    "Non-Pathological": NONPATHO_CLS_TASKS,
}
SEG_TASK_GROUPS = {
    "Pathological": PATHO_SEG_TASKS,
    "Anatomical": ANATOMICAL_SEG_TASKS,
}
PARADIGM_DISPLAY_NAMES = {
    "full_finetuning": "full fine-tuning",
    "head_only": "head-only training",
}
TABLE_SPECS = [
    ("classification", classification_summary, CLS_TASK_GROUPS, "macro F1"),
    ("segmentation", segmentation_summary, SEG_TASK_GROUPS, "positive Dice"),
]

for task_type, summary, task_groups, metric in TABLE_SPECS:
    for paradigm, paradigm_name in PARADIGM_DISPLAY_NAMES.items():
        frame = build_table_frame(summary, paradigm, task_groups)
        latex = to_booktabs_latex(
            frame,
            caption=(
                f"The mean and standard deviation of the {metric} score over three "
                f"seeds for each {task_type} task after {paradigm_name}. The best "
                "result for each task is shown in bold."
            ),
            label=f"tab:{task_type}_{paradigm}",
        )
        Path(f"{task_type}_{paradigm}_results.tex").write_text(latex)
        print(latex)